# 🎯 Activation Addition (ActAdd) with pyvene

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/)

**Steer Language Model Outputs at Inference Time Without Training**

This notebook demonstrates the **Activation Addition (ActAdd)** technique using the `pyvene` library. ActAdd allows you to control model outputs by adding "steering vectors" to the model's internal activations.

## 📚 What is ActAdd?

ActAdd is an **inference-time intervention** technique that:
1. Computes a "steering vector" from contrasting prompts (e.g., "Love" vs "Hate")
2. Adds this vector to the model's residual stream during generation
3. Steers outputs toward desired properties (sentiment, topic, style)

**Key Benefits:**
- ✅ No training required
- ✅ Works with a single pair of prompts
- ✅ Fast iteration over steering strategies
- ✅ Preserves performance on off-target tasks

## 📖 Reference

Based on: *"Activation Addition: Steering Language Models Without Optimization"*


## 1️⃣ Setup & Installation


In [ ]:
# Install pyvene from GitHub (includes ActAdd utilities)
%pip install -q git+https://github.com/stanfordnlp/pyvene.git
%pip install -q transformers accelerate


In [ ]:
import torch
import pyvene as pv
from transformers import AutoModelForCausalLM, AutoTokenizer

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Disable gradients for inference
torch.set_grad_enabled(False)


## 2️⃣ Load Model

We'll use GPT-2 XL for this demo. For Kaggle with GPU, you can also use:
- `meta-llama/Llama-2-7b-hf` (requires auth)
- `mistralai/Mistral-7B-v0.1`
- `facebook/opt-1.3b`


In [ ]:
# Load model and tokenizer
MODEL_NAME = "gpt2-xl"  # Change to your preferred model

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

# Set pad token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded! Layers: {pv.get_num_layers(model)}")


## 3️⃣ Compute Steering Vector

The core of ActAdd: we compute a steering vector by **contrasting** two prompts.

$$\text{steering\_vector} = \text{activation}(\text{"Love"}) - \text{activation}(\text{"Hate"})$$

This captures the "direction" of positivity in the model's representation space.


In [ ]:
# Define contrasting prompts
POSITIVE_PROMPT = "Love"
NEGATIVE_PROMPT = "Hate"

# Choose layer (typically 15-25% into the model works well)
LAYER = 6  # For GPT-2 XL (48 layers), try layers 6-20

# Compute steering vector
steering_vector = pv.compute_steering_vector(
    model=model,
    tokenizer=tokenizer,
    positive_prompt=POSITIVE_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    layer=LAYER,
    coeff=1.0,  # Default coefficient, we'll scale later
)

print(steering_vector)


## 4️⃣ Generate with Steering

Now let's see ActAdd in action! We'll generate text with and without the steering vector.


In [ ]:
# Test prompt
PROMPT = "I hate you because"

# Generate with different steering strengths
COEFFICIENTS = [0, 2, 5, 10]

print(f"Prompt: '{PROMPT}'\n")
print("=" * 80)

for coeff in COEFFICIENTS:
    if coeff == 0:
        # Unsteered generation
        inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
        output_ids = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            temperature=0.9,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
        output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    else:
        # Steered generation
        unsteered, output_text = pv.generate_with_steering(
            model=model,
            tokenizer=tokenizer,
            prompt=PROMPT,
            steering_vector=steering_vector,
            coeff=coeff,
            max_new_tokens=40,
            temperature=0.9,
            top_p=0.9,
        )
    
    print(f"\n[coeff={coeff}]")
    print(output_text)


## 5️⃣ Batch Comparison

Let's systematically compare multiple samples across different prompts.


In [ ]:
# Test on multiple prompts
TEST_PROMPTS = [
    "I hate you because",
    "I want to kill you because",
    "The worst thing about today is",
]

# Generate comparisons
results = pv.compare_generations(
    model=model,
    tokenizer=tokenizer,
    prompts=TEST_PROMPTS,
    steering_vector=steering_vector,
    coeff=5.0,
    num_samples=2,
    max_new_tokens=40,
    temperature=0.9,
)

# Print results nicely
pv.print_comparison(results, prompt_only=True)


## 6️⃣ Explore Different Steering Vectors

ActAdd is not limited to sentiment! Let's explore other steering directions.


In [ ]:
# Define different steering concepts
STEERING_CONCEPTS = [
    ("Love", "Hate", "Sentiment: Love → Hate"),
    ("Happy", "Sad", "Mood: Happy → Sad"),
    ("The Eiffel Tower is in Rome", "The Eiffel Tower is in France", "False Belief"),
    ("Intent to praise", "Intent to hurt", "Intent Steering"),
    ("I talk about weddings constantly", "I never talk about weddings", "Topic: Weddings"),
]

# Compute all steering vectors
steering_vectors = {}
for pos, neg, name in STEERING_CONCEPTS:
    sv = pv.compute_steering_vector(
        model=model,
        tokenizer=tokenizer,
        positive_prompt=pos,
        negative_prompt=neg,
        layer=LAYER,
    )
    steering_vectors[name] = sv
    print(f"✓ Computed: {name}")


In [ ]:
# Test each steering vector
PROMPT = "I went up to my friend and said"

print(f"Prompt: '{PROMPT}'\n")
print("=" * 80)

# First show unsteered
inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
output_ids = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.9,
    pad_token_id=tokenizer.pad_token_id,
)
print(f"\n[UNSTEERED]")
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# Show each steering vector
for name, sv in steering_vectors.items():
    _, steered = pv.generate_with_steering(
        model=model,
        tokenizer=tokenizer,
        prompt=PROMPT,
        steering_vector=sv,
        coeff=5.0,
        max_new_tokens=40,
        temperature=0.9,
    )
    print(f"\n[{name}]")
    print(steered)


## 7️⃣ Multi-Layer Steering

You can apply steering vectors at multiple layers simultaneously for more complex effects.


In [ ]:
# Compute steering vectors at different layers
sv_layer_6 = pv.compute_steering_vector(
    model=model,
    tokenizer=tokenizer,
    positive_prompt="Love",
    negative_prompt="Hate",
    layer=6,
    coeff=3.0,
)

sv_layer_12 = pv.compute_steering_vector(
    model=model,
    tokenizer=tokenizer,
    positive_prompt="Happy",
    negative_prompt="Sad",
    layer=12,
    coeff=3.0,
)

# Create multi-layer ActAdd model
multi_actadd = pv.create_multi_layer_actadd_model(
    model=model,
    steering_vectors=[sv_layer_6, sv_layer_12],
    coeff=1.0,  # Additional global scaling
)

# Generate
PROMPT = "I feel terrible because"
inputs = tokenizer(PROMPT, return_tensors="pt").to(device)

_, output_ids = multi_actadd.generate(
    inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.9,
    pad_token_id=tokenizer.pad_token_id,
)

print(f"Prompt: '{PROMPT}'")
print(f"Multi-layer steered output:")
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))


## 8️⃣ Understanding Layer Effects

Different layers capture different aspects. Let's visualize how the same steering vector affects output at different layers.


In [ ]:
# Test steering at different layers
PROMPT = "I hate you because"
num_layers = pv.get_num_layers(model)
LAYERS_TO_TEST = [0, 6, 12, num_layers // 2, num_layers - 12, num_layers - 1]

print(f"Prompt: '{PROMPT}'")
print(f"Steering: 'Love' - 'Hate' (coeff=5.0)\n")
print("=" * 80)

for layer in LAYERS_TO_TEST:
    if layer < 0 or layer >= num_layers:
        continue
    try:
        sv = pv.compute_steering_vector(
            model=model,
            tokenizer=tokenizer,
            positive_prompt="Love",
            negative_prompt="Hate",
            layer=layer,
        )
        
        _, steered = pv.generate_with_steering(
            model=model,
            tokenizer=tokenizer,
            prompt=PROMPT,
            steering_vector=sv,
            coeff=5.0,
            max_new_tokens=30,
            temperature=0.9,
        )
        
        # Show just the generated part
        generated = steered[len(PROMPT):]
        print(f"Layer {layer:2d}: {generated[:60]}...")
    except Exception as e:
        print(f"Layer {layer:2d}: Error - {e}")


## 9️⃣ Interactive Playground

Try your own steering experiments!


In [ ]:
# ===== CUSTOMIZE THESE =====
POSITIVE = "Peace and harmony"  # What you want MORE of
NEGATIVE = "War and conflict"   # What you want LESS of
LAYER = 10                       # Which layer (0 to num_layers-1)
COEFF = 5.0                      # Steering strength (try 1-15)
PROMPT = "The future of humanity is"  # Your test prompt
# ===========================

# Compute and apply
sv = pv.compute_steering_vector(
    model=model,
    tokenizer=tokenizer,
    positive_prompt=POSITIVE,
    negative_prompt=NEGATIVE,
    layer=LAYER,
)

unsteered, steered = pv.generate_with_steering(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    steering_vector=sv,
    coeff=COEFF,
    max_new_tokens=60,
    temperature=0.9,
)

print("🔵 UNSTEERED:")
print(unsteered)
print("\n🟢 STEERED:")
print(steered)


## 🔬 Advanced: Using pyvene Directly

For more control, you can use pyvene's lower-level APIs.


In [ ]:
# Manual approach using pyvene primitives
from pyvene import (
    IntervenableModel,
    IntervenableConfig,
    RepresentationConfig,
    AdditionIntervention,
)

# Get the raw steering vector tensor
steering_tensor = steering_vector.scaled(5.0)  # Apply coefficient

# Create configuration manually
config = IntervenableConfig(
    representations=[
        RepresentationConfig(
            layer=steering_vector.layer,
            component="block_output",
            unit="pos",
            max_number_of_units=steering_tensor.shape[1],
            source_representation=steering_tensor,  # Constant source!
        )
    ],
    intervention_types=AdditionIntervention,
)

# Create intervenable model
intervenable = IntervenableModel(config, model)

# Generate
inputs = tokenizer("I hate you because", return_tensors="pt").to(device)

_, output = intervenable.generate(
    inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.9,
    pad_token_id=tokenizer.pad_token_id,
)

print(tokenizer.decode(output[0], skip_special_tokens=True))


## 📝 Summary

### Key Takeaways:

1. **ActAdd is simple**: Just compute `activation(positive) - activation(negative)` and add it during inference

2. **No training needed**: Works with a single pair of contrasting prompts

3. **Key hyperparameters**:
   - `layer`: Earlier layers (~0-30% of model) often work better for sentiment
   - `coeff`: Start with 1-5, increase for stronger effects (too high = gibberish)
   
4. **Composable**: Stack multiple steering vectors at different layers

### API Reference:

```python
# Compute steering vector
sv = pv.compute_steering_vector(model, tokenizer, "Love", "Hate", layer=6)

# Quick generation comparison
unsteered, steered = pv.generate_with_steering(model, tokenizer, prompt, sv, coeff=5)

# Create reusable ActAdd model
actadd = pv.create_actadd_model(model, sv, coeff=5)
_, output = actadd.generate(inputs, max_new_tokens=50)

# Multi-layer steering
multi = pv.create_multi_layer_actadd_model(model, [sv1, sv2], coeff=3)
```

### Further Reading:
- [pyvene Documentation](https://github.com/stanfordnlp/pyvene)
- [Original ActAdd Paper](https://arxiv.org/abs/2308.10248)
